In [8]:
import pandas as pd
from umap import UMAP
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

## BERTopic

### Setup

In [9]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=20, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english")
ctfidf_model = ClassTfidfTransformer()
representation_model = KeyBERTInspired()

topic_model = BERTopic(
  embedding_model=embedding_model,          # Extract embeddings
  umap_model=umap_model,                    # Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Tokenize topics
  ctfidf_model=ctfidf_model,                # Extract topic words
  representation_model=representation_model # Fine-tune topic representations
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
df = pd.read_excel("../data/interim/clean_lines.xlsx")
docs = df['texto_limpo'].tolist()
timestamps = df['created_at'].tolist()

### Training

In [11]:
topics, probs = topic_model.fit_transform(docs)

### Loading

In [5]:
topic_model = BERTopic.load("../models/saves/min 20", embedding_model="all-MiniLM-L6-v2")
topic_model.transform(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(array([ 67, 125, 164, ...,   9,   0, 118], shape=(22872,)),
 array([0.81121045, 0.5089471 , 0.8364431 , ..., 0.6749325 , 0.6851254 ,
        0.66205794], shape=(22872,), dtype=float32))

## Model Evaluation

In [12]:
info = topic_model.get_topic_info()
n_topicos = (info.Topic != -1).sum()
pct_outliers = info.loc[info.Topic == -1, "Count"].sum() / info.Count.sum()
print(n_topicos, f"{pct_outliers:.1%}")

reais = info[info.Topic != -1]
print(reais.Count.describe())
print(reais.nlargest(5, "Count")[["Topic", "Count", "Name"]])

198 35.9%
count    198.000000
mean      74.030303
std       73.152570
min       20.000000
25%       30.250000
50%       51.000000
75%       81.250000
max      551.000000
Name: Count, dtype: float64
   Topic  Count                                               Name
1      0    551                     0_recession_wage_fiscal_yields
2      1    432               1_demand_deficit_quarterly_estimates
3      2    427  2_outperformance_underperformance_decline_markets
4      3    319                               3_rates_euro_25bp_eu
5      4    271           4_forecasts_forecast_consensus_economics


In [ ]:
import numpy as np
from itertools import combinations
from sklearn.feature_extraction.text import CountVectorizer

def npmi_diversidade(topic_model, docs, top_n=10):
    ids = [t for t in topic_model.get_topics() if t != -1]
    palavras = {t: [w for w, _ in topic_model.get_topic(t)[:top_n]] for t in ids}
    vocab = sorted({w for ws in palavras.values() for w in ws})
    idx = {w: i for i, w in enumerate(vocab)}

    cv = CountVectorizer(vocabulary=vocab, ngram_range=(1, 3), binary=True)
    X = cv.transform(docs)                       # documentos x palavras (presença)
    N = X.shape[0]
    p = np.asarray(X.sum(axis=0)).ravel() / N
    co = (X.T @ X).toarray() / N                 # co-ocorrência por documento

    scores = []
    for t in ids:
        pares = []
        for a, b in combinations(palavras[t], 2):
            i, j = idx[a], idx[b]
            if p[i] == 0 or p[j] == 0:
                continue
            pij = co[i, j]
            pares.append(-1.0 if pij == 0 else np.log(pij / (p[i] * p[j])) / -np.log(pij))
        if pares:
            scores.append(np.mean(pares))

    flat = [w for ws in palavras.values() for w in ws]
    return {"npmi": round(float(np.mean(scores)), 3),
            "diversidade": round(len(set(flat)) / len(flat), 3)}

print(npmi_diversidade(topic_model, docs))

{'npmi': -0.047, 'diversidade': 0.361}
{'npmi': 0.283, 'diversidade': 0.621}


## Merge

In [ ]:
merge_groups = [
    [170,175], [36,129,141], [58,143,147,174,105], [3,125], [40,69,103], 
    [139,163], [31,64], [144,184], [162,177,37], [15,99], [73,161], 
    [18,171], [83,121], [59,173], [166,101], [108,131,134], [65,181],
    [98,154,54,120], [71,155], [116,44], [62,60], [185,35,66,137], [1,12],
    [33,17], [23,149,178], [55,157], [127,91], [30,183], [84,75,85], 
    [76,88], [0,112], [176,118], [4,94,124], [13,80], [21,68], [5,107],
    [97,89], [67,179], [77,135], [56,82], [106,133], [7,22,41], [11,47]
]

import networkx as nx
G = nx.Graph()
for g in merge_groups:
    G.add_edges_from(zip(g, g[1:]))
final_groups = [sorted(c) for c in nx.connected_components(G)]

topic_model.merge_topics(docs, final_groups)

## Visualization

### Topics Over Time

In [ ]:
topics_over_time = topic_model.topics_over_time(docs, timestamps)

2026-11-13 19:41:12,216 - BERTopic - WARNING: There are more than 100 unique timestamps (i.e., 491) which significantly slows down the application. Consider setting `nr_bins` to a value lower than 100 to speed up calculation. 


In [ ]:
topic_model.visualize_topics()

In [18]:
topic_model.visualize_heatmap()

In [ ]:
info = topic_model.get_topic_info()
freq = topic_model.get_topic_info()
top_ids = freq[freq.Topic != -1]["Topic"].tolist()

topic_model.visualize_barchart(top_n_topics=20, n_words=8, width=300, height=300)

fig = topic_model.visualize_topics_over_time(
    topics_over_time,
    topics=top_ids,        # só os 15 maiores
    normalize_frequency=True,
)

fig

In [ ]:
hierarquical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarquical_topics)

100%|██████████| 134/134 [00:09<00:00, 14.49it/s]


In [ ]:
topicos_validos = [t for t in topic_model.get_topics().keys() if t != -1]
matriz_sim = topic_model.approximate_distribution(docs)[0]

from sklearn.metrics.pairwise import cosine_similarity
embeddings_topicos = topic_model.topic_embeddings_[1:]
matriz_sim = cosine_similarity(embeddings_topicos)

df_sim = pd.DataFrame(matriz_sim, index=topicos_validos, columns=topicos_validos)

pares = df_sim.unstack().reset_index()
pares.columns = ['Topico_A', 'Topico_B', 'Similaridade']
pares_filtrados = pares[pares['Topico_A'] < pares['Topico_B']]

top_similares = pares_filtrados.sort_values(by='Similaridade', ascending=False)

print(top_similares.head(20))

       Topico_A  Topico_B  Similaridade
3498         24        42      0.911454
8022         55       102      0.902809
10741        74        85      0.891255
6103         42        55      0.887030
3558         24       102      0.884375
6150         42       102      0.881615
18569       128       137      0.878158
219           1        75      0.872991
151           1         7      0.868219
12339        85        99      0.867242
10322        71        98      0.862356
3511         24        55      0.857885
2486         17        38      0.850815
5222         36        38      0.846274
379           2        91      0.834593
1083          7        75      0.834027
7406         51        62      0.832820
9919         68       127      0.831768
12212        84       116      0.830490
269           1       125      0.830443


In [28]:
import plotly.graph_objects as go

# tabela período x tópico
t = topics_over_time[topics_over_time.Topic != -1]
tab = (t.pivot_table(index="Timestamp", columns="Topic",
                     values="Frequency", aggfunc="sum", fill_value=0)
         .sort_index())

tab.index = pd.to_datetime(tab.index)
tab = tab.resample("W").sum()

# participação por período
share = tab.div(tab.sum(axis=1), axis=0)

# com memória
def com_memoria(df, meia_vida=6):
    lam = 0.5 ** (1 / meia_vida)
    out = df.astype(float).copy()
    for i in range(1, len(df)):
        out.iloc[i] = lam * out.iloc[i - 1].values + df.iloc[i].values
    return out

nomes = dict(zip(info.Topic, info.Name))
top_ids = [t for t in top_ids if t in share.columns and t in nomes]

lam = 0.5 ** (1 / 6)
estoque = com_memoria(share[top_ids], meia_vida=6) * (1 - lam)

# gráfico
nomes = info.set_index("Topic").Name
fig2 = go.Figure()
for tid in top_ids:
    fig2.add_trace(go.Scatter(x=estoque.index, y=estoque[tid],
                              mode="lines", name=nomes[tid]))
fig2.update_layout(title="Evolução acumulada dos tópicos ao longo do tempo (com memória)",
                   width=1000, height=500)
fig2.show()

In [ ]:
a = pd.DataFrame({
    "date": pd.to_datetime(timestamps),
    "topic": topic_model.topics_,
    "msg": df["texto_limpo"].tolist()
})
d = a[a.topic != -1]
tabw = (d.groupby([pd.Grouper(key="date", freq="W"), "topic"])
          .size().unstack(fill_value=0))

print(tabw.sum(axis=1).describe())

count     87.000000
mean     168.482759
std       50.348521
min       48.000000
25%      138.000000
50%      172.000000
75%      200.000000
max      266.000000
dtype: float64


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.influencia_topicos import *

pares_sob, grupos = sobreposicao(topic_model, limiar=0.80)
cont, share, clr  = preparar(a, grupos, freq="W")
topicos = maiores(cont, n=15)

subs = substituicao(clr, topicos)
ante = antecedencia(clr, topicos, max_lag=3)
coo  = coocorrencia(a, grupos)

print(coo)

[info] 87 períodos, 117 tópicos/grupos.
{'mais_associados': Empty DataFrame
Columns: [A, B, relatorios_juntos, lift, npmi]
Index: [], 'menos_associados': Empty DataFrame
Columns: [A, B, relatorios_juntos, lift, npmi]
Index: []}


In [1]:
from scipy.stats import poisson
import numpy as np

HL_BASE = 6
BURN = 8

tot = tabw.sum(axis=1)
share = tabw.div(tot.replace(0, np.nan), axis=0).fillna(0)

base = share.ewm(halflife=3, adjust=False).mean().shift(1)
expected = base.mul(tot, axis=0) + 0.

# surpresa = -log10(p)
p = poisson.sf(tabw.values - 1, expected.values).clip(1e-300)
surpresa = pd.DataFrame(-np.log10(p), index=tabw.index, columns=tabw.columns).iloc[BURN:]

# topicos com maior pico de surpresa
nomes = info.set_index("Topic").Name
top_ids = [t for t in surpresa.max().index if t in nomes.index]

# grafico
fig = go.Figure()
for tid in top_ids:
    fig.add_trace(go.Scatter(x=surpresa.index, y=surpresa[tid],
                             mode="lines", name=nomes[tid]))
fig.update_layout(title="Surpresa dos tópicos ao longo do tempo (-log10 p, Poisson)",
                  yaxis_title="surpresa", width=1000, height=500)
fig.show()


NameError: name 'tabw' is not defined